## Calculate measurement averages and aggregate results into CSV file

In [ ]:
"""
Tomogram Analysis Pipeline - Measurement Aggregation

This notebook processes biological measurement data to:
1. Calculate averages for individual tomogram folders
2. Merge all averages into a single comprehensive CSV file

"""

# Library imports
import os
import re
import pandas as pd

### Individual Tomogram Averages

In [ ]:
def calculate_individual_averages(directory):
    """
    Calculate average measurements for each tomogram folder containing '_measurements.csv' files.
    
    Processes only immediate subdirectories of 'measurements2' folders and saves results
    as '_average_values.csv' in each respective folder.
    
    Args:
        directory (str): Root directory to search for measurement files
    """
    for root, dirs, files in os.walk(directory):
        if 'measurements2' not in root.split(os.path.sep):
            continue

        path_parts = root.split(os.path.sep)
        if 'measurements2' in path_parts:
            measurements2_index = path_parts.index('measurements2')
            if len(path_parts) != measurements2_index + 2:
                continue  # Skip if not an immediate child

        for file in files:
            if file.endswith('_measurements.csv'):
                print(f"Processing {file} in {root}")
                    
                input_csv = os.path.join(root, file)
                folder_name = os.path.basename(root)
                output_csv = os.path.join(root, f'{folder_name}_average_values.csv')
                
                df = pd.read_csv(input_csv)
                
                avg_convex_hull_volume = df['volume - convex hull '].mean()
                avg_regionprops_volume = df['volume - regionprops'].mean()
                avg_max_diameter = df['max_diameter'].mean()
                avg_sphericity = df['Sphericity'].mean()
                avg_median_2d_feret_diameter = df['median_2d_feret_diameter'].mean()
                avg_sphere_volume_equivalent = df['sphere_volume_equivalent'].mean()
                avg_mean_2d_feret_diameter = df['mean_2d_feret_diameter'].mean()

                std_convex_hull_volume = df['volume - convex hull '].std()
                std_median_2d_feret_diameter = df['median_2d_feret_diameter'].std()
                std_sphere_volume_equivalent = df['sphere_volume_equivalent'].std()
                std_sphericity = df['Sphericity'].std()
                      
                num_components = len(df)
                
                
                averages_df = pd.DataFrame({
                    'avg_convex_hull_volume': [avg_convex_hull_volume],
                    'avg_regionprops_volume': [avg_regionprops_volume],
                    'avg_max_diameter': [avg_max_diameter],
                    'num_components': [num_components],
                    'avg_sphericity': [avg_sphericity],
                    'avg_median_2d_feret_diameter': [avg_median_2d_feret_diameter],
                    'avg_mean_2d_feret_diameter': [avg_mean_2d_feret_diameter],
                    'avg_sphere_volume_equivalent': [avg_sphere_volume_equivalent],
                    'std_convex_hull_volume': [std_convex_hull_volume],
                    'std_sphere_volume_equivalent': [std_sphere_volume_equivalent],
                    'std_median_2d_feret_diameter': [std_median_2d_feret_diameter],
                    'std_sphericity': [std_sphericity],
                })
                
                
                averages_df.to_csv(output_csv, index=False)



### Merge results into one CSV file 

In [ ]:
def merge_csv_files(main_directory):
    """
    Merge all '_average_values.csv' files from subdirectories into one comprehensive CSV.
    
    Args:
        main_directory (str): Root directory containing all measurement folders
    """
    all_data = []

    for root, dirs, files in os.walk(main_directory):
        
        if 'measurements2' not in root.split(os.path.sep):
            continue
       
        path_parts = root.split(os.path.sep)
        if 'measurements2' in path_parts:
            measurements2_index = path_parts.index('measurements2')
            if len(path_parts) != measurements2_index + 2:
                continue  # Skip if not an immediate child

        for file in files:
            if file.endswith('_average_values.csv'):
                file_path = os.path.join(root, file)
                df = pd.read_csv(file_path)
                
                folder_name = os.path.basename(root)
                folder_number = int(re.search(r'\d+', folder_name).group())
                  
                subfolder_parts = root.split(os.sep)
                if 'whopper2' in subfolder_parts:
                    subfolder_name = 'whopper2'  # Special case for whopper2
                else:
                    subfolder_name = f"{subfolder_parts[-4]} {subfolder_parts[-3]}"
                
                # Add metadata columns
                df.insert(0, 'subfolder_name', subfolder_name)
                df.insert(1, 'tomogram_number', folder_name)
                df['folder_number'] = folder_number
                
                all_data.append(df)
    
    
    if all_data:  
        merged_df = pd.concat(all_data, ignore_index=True)
        
        
        merged_df.sort_values(by=['subfolder_name', 'tomogram_number'], inplace=True)
        merged_df.drop(columns=['folder_number'], inplace=True)
        
        
        output_csv = os.path.join(main_directory, 'all_averages_wmean_and_std.csv')
        merged_df.to_csv(output_csv, index=False)
        print(f"Successfully merged data to {output_csv}")
    else:
        print("Warning: No average files found to merge")






In [ ]:
# Execution


if __name__ == "__main__":
    # Root directory 
    main_directory = 'sv_measurements/'

    calculate_individual_averages(main_directory)
    merge_csv_files(main_directory)